# 스마트 창고 출고 지연 예측 — Model v7 (Sequence Model)

**접근 방식 전환**

| | GBDT (v5/v6) | v7 시퀀스 모델 |
|--|--|--|
| 입력 | 행 단위 (1 타임스텝) | 시나리오 단위 (25 타임스텝 전체) |
| 타겟 lag | 사용 (test에서 오차 누적) | **미사용** (test에서 불필요) |
| 추론 방식 | Autoregressive | **직접 예측 (25개 동시)** |
| OOF-LB 갭 | ~3.7 | 없음 |

**모델**: LSTM + Transformer 학습 후 앙상블  
**실행 환경**: Google Colab T4 GPU

## 0. 환경 설정

In [ ]:
!pip install -q lightgbm xgboost catboost optuna

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import math
from torch.utils.data import Dataset, DataLoader
import lightgbm as lgb
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
import optuna
from optuna.samplers import TPESampler
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

DEVICE  = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED    = 42
N_FOLD  = 5
N_STEPS = 25
TARGET  = 'avg_delay_minutes_next_30m'
ID_COLS = ['ID', 'layout_id', 'scenario_id']

torch.manual_seed(SEED)
np.random.seed(SEED)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

print(f'Device: {DEVICE}')
print(f'GPU: {torch.cuda.get_device_name(0) if DEVICE=="cuda" else "없음"}')

## 1. Google Drive 마운트 & 데이터 로드

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/warehouse/'

train  = pd.read_csv(DATA_DIR + 'train.csv')
test   = pd.read_csv(DATA_DIR + 'test.csv')
layout = pd.read_csv(DATA_DIR + 'layout_info.csv')
print(f'train: {train.shape} / test: {test.shape}')

## 2. 전처리

In [ ]:
train = train.merge(layout, on='layout_id', how='left')
test  = test.merge(layout,  on='layout_id', how='left')

le = LabelEncoder()
train['layout_type'] = le.fit_transform(train['layout_type'].astype(str))
test['layout_type']  = le.transform(test['layout_type'].astype(str))

train = train.sort_values(['scenario_id', 'shift_hour']).reset_index(drop=True)
test  = test.sort_values(['scenario_id',  'shift_hour']).reset_index(drop=True)

ts_features = [
    'order_inflow_15m', 'robot_utilization', 'congestion_score',
    'fault_count_15m',  'loading_dock_util', 'battery_mean',
    'blocked_path_15m', 'task_reassign_15m', 'low_battery_ratio',
    'avg_trip_distance',
]

def make_ts_features(df, cols, group_col='scenario_id'):
    df = df.copy()
    grp = df.groupby(group_col)
    for col in cols:
        for lag in [1, 2, 3]:
            df[f'{col}_lag{lag}'] = grp[col].shift(lag)
        for win in [3, 5]:
            df[f'{col}_roll_mean{win}'] = grp[col].transform(
                lambda x: x.shift(1).rolling(win, min_periods=1).mean())
            df[f'{col}_roll_std{win}'] = grp[col].transform(
                lambda x: x.shift(1).rolling(win, min_periods=2).std())
        df[f'{col}_cumsum'] = grp[col].transform(lambda x: x.shift(1).cumsum())
    return df

def make_extra_features(df):
    df = df.copy()
    df['time_step']          = df.groupby('scenario_id').cumcount()
    df['robot_active_ratio'] = df['robot_active'] / (
        df['robot_active'] + df['robot_idle'] + df['robot_charging'] + 1e-6)
    df['battery_congestion'] = df['low_battery_ratio'] * df['congestion_score']
    df['urgent_volume']      = df['urgent_order_ratio'] * df['order_inflow_15m']
    df['fault_impact']       = df['fault_count_15m'] * df['avg_recovery_time']
    df['dock_stress']        = df['outbound_truck_wait_min'] * df['loading_dock_util']
    return df

train = make_ts_features(train, ts_features)
test  = make_ts_features(test,  ts_features)
train = make_extra_features(train)
test  = make_extra_features(test)

# 시퀀스 모델용: 타겟 lag 없이 입력 피처만 사용
feat_cols = [c for c in train.columns if c not in ID_COLS + [TARGET]]
X         = train[feat_cols].values
y         = train[TARGET].values
X_test    = test[feat_cols].values
groups    = train['scenario_id']

# NaN → median + StandardScaler
imputer = SimpleImputer(strategy='median')
scaler  = StandardScaler()
X         = scaler.fit_transform(imputer.fit_transform(X)).astype(np.float32)
X_test    = scaler.transform(imputer.transform(X_test)).astype(np.float32)
y         = y.astype(np.float32)

print(f'피처 수: {len(feat_cols)} | X: {X.shape}')

## 3. 시퀀스 데이터 준비

각 시나리오(25 타임스텝)를 하나의 시퀀스로 변환

In [ ]:
def to_sequences(X_flat, y_flat, scenario_arr, step_arr, unique_sc):
    sc_to_idx = {sc: i for i, sc in enumerate(unique_sc)}
    sc_idx    = np.array([sc_to_idx[sc] for sc in scenario_arr])
    step_idx  = step_arr.astype(int)
    n, f      = len(unique_sc), X_flat.shape[1]

    X_seq = np.zeros((n, N_STEPS, f), dtype=np.float32)
    X_seq[sc_idx, step_idx] = X_flat

    y_seq = None
    if y_flat is not None:
        y_seq = np.zeros((n, N_STEPS), dtype=np.float32)
        y_seq[sc_idx, step_idx] = y_flat
    return X_seq, y_seq


train_steps = train.groupby('scenario_id').cumcount().values
test_steps  = test.groupby('scenario_id').cumcount().values
unique_train_sc = pd.unique(train['scenario_id'])
unique_test_sc  = pd.unique(test['scenario_id'])

X_seq_train, y_seq_train = to_sequences(
    X, y, train['scenario_id'].values, train_steps, unique_train_sc)
X_seq_test, _ = to_sequences(
    X_test, None, test['scenario_id'].values, test_steps, unique_test_sc)

print(f'train seq: {X_seq_train.shape}')  # (10000, 25, features)
print(f'test  seq: {X_seq_test.shape}')   # (2000,  25, features)

## 4. 모델 정의

### 4-1. LSTM

In [ ]:
class WarehouseLSTM(nn.Module):
    def __init__(self, input_size, hidden=256, n_layers=3, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden, n_layers,
                            batch_first=True, dropout=dropout)
        self.head = nn.Sequential(
            nn.Linear(hidden, 128), nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):             # (B, 25, F)
        out, _ = self.lstm(x)         # (B, 25, hidden)
        return self.head(out).squeeze(-1)  # (B, 25)


print('LSTM 정의 완료')

### 4-2. Transformer

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=25, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, 25, d_model)

    def forward(self, x):
        return self.dropout(x + self.pe)


class WarehouseTransformer(nn.Module):
    def __init__(self, input_size, d_model=256, nhead=8, n_layers=4,
                 dim_ff=512, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(input_size, d_model)
        self.pos_enc    = PositionalEncoding(d_model, dropout=dropout)
        enc_layer = nn.TransformerEncoderLayer(
            d_model, nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.Linear(d_model, 128), nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):                          # (B, 25, F)
        x = self.pos_enc(self.input_proj(x))       # (B, 25, d_model)
        x = self.transformer(x)                    # (B, 25, d_model)
        return self.head(x).squeeze(-1)            # (B, 25)


print('Transformer 정의 완료')

## 5. 학습 유틸리티

In [ ]:
class SeqDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y) if y is not None else None
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        return (self.X[i], self.y[i]) if self.y is not None else self.X[i]


def train_model(model, X_tr, y_tr, X_val, y_val,
                epochs=100, patience=15, lr=1e-3, batch_size=256):
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit  = nn.MSELoss()  # RMSE 직접 최적화 (대회 평가지표 정렬)

    tr_loader  = DataLoader(SeqDataset(X_tr,  y_tr),  batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(SeqDataset(X_val, y_val), batch_size=batch_size)

    best_val, best_state, no_imp = np.inf, None, 0
    for epoch in range(epochs):
        model.train()
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            crit(model(xb), yb).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()

        model.eval()
        with torch.no_grad():
            val_pred = torch.cat(
                [model(xb.to(DEVICE)) for xb, _ in val_loader]
            ).cpu().numpy()
        val_rmse = np.sqrt(np.mean((val_pred - y_val) ** 2))

        if val_rmse < best_val:
            best_val, best_state, no_imp = val_rmse, model.state_dict(), 0
        else:
            no_imp += 1
            if no_imp >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_val


def predict_seq(model, X_seq, batch_size=256):
    """(n_sc, 25, F) → (n_sc, 25) 예측"""
    model.eval()
    loader = DataLoader(SeqDataset(X_seq), batch_size=batch_size)
    with torch.no_grad():
        preds = torch.cat([model(xb.to(DEVICE)) for xb in loader]).cpu().numpy()
    return preds


def seq_pred_to_flat(pred_seq, scenario_arr, step_arr, unique_sc):
    """(n_sc, 25) 예측값 → 원래 행 순서로 복원"""
    sc_to_idx = {sc: i for i, sc in enumerate(unique_sc)}
    sc_idx    = np.array([sc_to_idx[sc] for sc in scenario_arr])
    step_idx  = step_arr.astype(int)
    return pred_seq[sc_idx, step_idx]


print('학습 유틸리티 정의 완료')

## 6. LSTM Group K-Fold 학습

In [ ]:
n_sc      = len(unique_train_sc)
input_dim = X_seq_train.shape[2]
gkf_sc    = GroupKFold(n_splits=N_FOLD)

lstm_oof_seq  = np.zeros((n_sc, N_STEPS))
lstm_pred_seq = np.zeros((len(unique_test_sc), N_STEPS))

for fold, (tr_idx, val_idx) in enumerate(
    gkf_sc.split(range(n_sc), range(n_sc), range(n_sc))
):
    model = WarehouseLSTM(input_dim).to(DEVICE)
    model, best_val = train_model(
        model,
        X_seq_train[tr_idx],  y_seq_train[tr_idx],
        X_seq_train[val_idx], y_seq_train[val_idx],
        epochs=150, patience=20, lr=5e-4
    )
    lstm_oof_seq[val_idx]  = predict_seq(model, X_seq_train[val_idx])
    lstm_pred_seq         += predict_seq(model, X_seq_test) / N_FOLD

    val_flat  = lstm_oof_seq[val_idx].flatten()
    true_flat = y_seq_train[val_idx].flatten()
    print(f'  [LSTM] Fold {fold+1}  RMSE: {rmse(true_flat, val_flat):.4f}  (best val: {best_val:.4f})')

# 시퀀스 → 행 단위 복원
lstm_oof  = seq_pred_to_flat(lstm_oof_seq,  train['scenario_id'].values, train_steps, unique_train_sc)
lstm_pred = seq_pred_to_flat(lstm_pred_seq, test['scenario_id'].values,  test_steps,  unique_test_sc)
lstm_score = rmse(y, lstm_oof)
print(f'\n  [LSTM] OOF RMSE: {lstm_score:.4f}')

## 7. Transformer Group K-Fold 학습

In [ ]:
tf_oof_seq  = np.zeros((n_sc, N_STEPS))
tf_pred_seq = np.zeros((len(unique_test_sc), N_STEPS))

for fold, (tr_idx, val_idx) in enumerate(
    gkf_sc.split(range(n_sc), range(n_sc), range(n_sc))
):
    model = WarehouseTransformer(input_dim).to(DEVICE)
    model, best_val = train_model(
        model,
        X_seq_train[tr_idx],  y_seq_train[tr_idx],
        X_seq_train[val_idx], y_seq_train[val_idx],
        epochs=150, patience=20, lr=5e-4
    )
    tf_oof_seq[val_idx]  = predict_seq(model, X_seq_train[val_idx])
    tf_pred_seq         += predict_seq(model, X_seq_test) / N_FOLD

    val_flat  = tf_oof_seq[val_idx].flatten()
    true_flat = y_seq_train[val_idx].flatten()
    print(f'  [TF] Fold {fold+1}  RMSE: {rmse(true_flat, val_flat):.4f}  (best val: {best_val:.4f})')

tf_oof  = seq_pred_to_flat(tf_oof_seq,  train['scenario_id'].values, train_steps, unique_train_sc)
tf_pred = seq_pred_to_flat(tf_pred_seq, test['scenario_id'].values,  test_steps,  unique_test_sc)
tf_score = rmse(y, tf_oof)
print(f'\n  [Transformer] OOF RMSE: {tf_score:.4f}')

## 8. Optuna 앙상블 가중치 최적화

In [ ]:
print('모델별 단독 OOF RMSE')
print(f'  LSTM       : {lstm_score:.4f}')
print(f'  Transformer: {tf_score:.4f}')

def ens_obj(trial):
    w1 = trial.suggest_float('w_lstm', 0.0, 1.0)
    w2 = trial.suggest_float('w_tf',   0.0, 1.0)
    s  = w1 + w2
    if s < 1e-6: return 1e9
    return rmse(y, (w1*lstm_oof + w2*tf_oof) / s)

study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED))
study.optimize(ens_obj, n_trials=500, show_progress_bar=True)

bw   = study.best_params
s    = bw['w_lstm'] + bw['w_tf']
w_l  = bw['w_lstm'] / s
w_t  = bw['w_tf']   / s

ens_oof  = w_l*lstm_oof  + w_t*tf_oof
ens_pred = w_l*lstm_pred + w_t*tf_pred
ens_score = rmse(y, ens_oof)

print(f'\n최적 가중치  LSTM: {w_l:.4f} | TF: {w_t:.4f}')
print(f'앙상블 OOF RMSE : {ens_score:.4f}')
print(f'v5(GBDT) 대비   : {20.1829 - ens_score:+.4f}  (LB 기준)')

## 9. GBDT OOF와 스태킹 (선택)

시퀀스 모델 단독 성능이 충분하면 스킵 가능

In [ ]:
# v6 best params (로컬에서 얻은 값)
# 시퀀스 모델 단독 성능 확인 후 스태킹 여부 결정
DO_STACKING = False   # 시퀀스 모델이 충분히 좋으면 False 유지

if DO_STACKING:
    # v6 best params를 여기에 붙여넣기
    lgb_best = {}  # v6 노트북 출력에서 복사
    xgb_best = {}
    cat_best = {}

    # 타겟 lag 포함 피처셋으로 재전처리 필요
    print('스태킹 모드: 별도 GBDT 학습 필요')
else:
    print('시퀀스 모델 단독 예측 사용')

## 10. 성능 비교 및 제출 파일 생성

In [ ]:
import matplotlib.pyplot as plt

print('=' * 55)
print('버전별 OOF RMSE 비교')
print('=' * 55)
results = {
    'v5 GBDT (타겟 lag, OOF)':  16.4661,
    'v6 GBDT (Optuna 재튜닝)':  16.4383,
    'v7 LSTM':                   lstm_score,
    'v7 Transformer':            tf_score,
    'v7 앙상블':                 ens_score,
}
for name, score in results.items():
    print(f'  {name:35s}: {score:.4f}')

print()
print('리더보드 기준 (OOF 불일치 주의)')
print(f'  v5 LB: 20.1829  |  v6 LB: 20.1974')
print(f'  v7 예상 LB: OOF-LB 갭 없으므로 OOF ≈ LB 기대')

colors = ['lightsteelblue', 'lightblue', 'steelblue', 'seagreen', 'tomato']
fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.barh(list(results.keys()), list(results.values()),
               color=colors, edgecolor='white')
for bar, score in zip(bars, results.values()):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f'{score:.4f}', va='center', fontsize=9)
ax.set_xlabel('OOF RMSE (낮을수록 좋음)')
ax.set_title('v7 Sequence Model vs GBDT', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
submission = pd.read_csv(DATA_DIR + 'sample_submission.csv')
submission[TARGET] = ens_pred
submission.to_csv(DATA_DIR + 'submission_v7.csv', index=False)

print(f'submission_v7.csv 저장 완료')
print(f'앙상블 OOF RMSE: {ens_score:.4f}')
print(submission[TARGET].describe().round(4))